In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# --------------------------------------------------
# Find project root automatically
# --------------------------------------------------

current = Path.cwd().resolve()

while not (current / "src").exists():
    if current.parent == current:
        raise FileNotFoundError("Could not locate project root.")
    current = current.parent

PROJECT_ROOT = current
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)

from agents.q_learning_agent import (
    QLearningAgent,
    QL_CONFIG
)

plt.style.use("seaborn-v0_8")

print("✅ Q-Learning modules loaded!")
print(f"Project Root : {PROJECT_ROOT}")
print(f"Source Path  : {SRC_PATH}")

print("\nQ-Learning Config:")
for k, v in QL_CONFIG.items():
    print(f"  {k:<20}: {v}")

In [ ]:
print("=" * 55)
print("  BELLMAN EQUATION EXPLANATION")
print("=" * 55)
print("""
Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]

Where:
  s  = current state (inventory, days_left)
  a  = current action (price level)
  r  = reward (revenue from sale)
  s' = next state
  α  = learning rate (0.1) — how fast to learn
  γ  = discount factor (0.99) — value future rewards

Example:
  State  : (30 tickets, 10 days left)
  Action : Set price $200
  Result : Customer buys → reward = $200
  
  Q(30,10,$200) gets updated to reflect
  that $200 is good here!
""")

In [ ]:
env   = DynamicPricingEnv()
agent = QLearningAgent(env, QL_CONFIG)

print("Training Q-Learning agent...")
print("5000 episodes — takes ~2 minutes\n")

rewards = agent.train(
    n_episodes=5000,
    verbose=True
)

print(f"\n✅ Training complete!")
print(f"   Final 100-ep mean: "
      f"${np.mean(rewards[-100:]):.0f}")

In [ ]:
from training.q_learning_trainer import (
    plot_training_curves
)

plot_training_curves(
    agent,
    save_path='../results/ql_training.png'
)

In [ ]:
from training.q_learning_trainer import (
    plot_q_table_heatmap
)

plot_q_table_heatmap(
    agent,
    save_path='../results/q_table_policy.png'
)

In [ ]:
# What did the agent learn?
price_policy = agent.get_price_policy()

print("=== LEARNED POLICY ANALYSIS ===\n")
print("Optimal prices at key states:")
print(f"{'State':<30} {'Optimal Price'}")
print("─" * 45)

key_states = [
    (50, 30, "Full inventory, lots of time"),
    (50, 5,  "Full inventory, urgent!"),
    (10, 30, "Low inventory, lots of time"),
    (10, 5,  "Low inventory, urgent!"),
    (25, 15, "Half inventory, half time"),
    (5,  2,  "Almost sold out, 2 days!"),
]

for inv, days, desc in key_states:
    price = price_policy[inv, days]
    print(f"  ({inv:2d} tickets, {days:2d} days) "
          f"{desc:<28}: ${price}")

In [ ]:
from agents.baseline_agents import (
    FixedPriceAgent,
    TimedPricingAgent,
    DemandBasedAgent,
    LinearDecayAgent
)
from utils.evaluator import evaluate_agent

print("Evaluating all agents...\n")

# Evaluate Q-Learning
ql_results = agent.evaluate(
    n_episodes=100, seed=42
)

# Evaluate baselines
baselines = {
    'Fixed Price'   : FixedPriceAgent(env),
    'Time Based'    : TimedPricingAgent(env),
    'Demand Based'  : DemandBasedAgent(env),
    'Linear Decay'  : LinearDecayAgent(env)
}

comparison = {'Q-Learning': ql_results['mean_revenue']}
for name, b_agent in baselines.items():
    df = evaluate_agent(b_agent, n_episodes=100)
    comparison[name] = df['total_revenue'].mean()
    print(f"  {name:<15}: ${comparison[name]:.0f}")

print(f"  {'Q-Learning':<15}: "
      f"${comparison['Q-Learning']:.0f} 🤖")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

agents_names = list(comparison.keys())
revenues = list(comparison.values())
colors = [
    'gold' if n == 'Q-Learning'
    else 'steelblue'
    for n in agents_names
]

bars = ax.bar(
    agents_names,
    revenues,
    color=colors,
    edgecolor='black',
    width=0.6
)
ax.set_title(
    'Q-Learning vs Baseline Agents\n'
    'Mean Revenue per Episode',
    fontweight='bold',
    fontsize=13
)
ax.set_ylabel('Mean Revenue ($)')
for bar, val in zip(bars, revenues):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        val + 50,
        f'${val:.0f}',
        ha='center',
        fontweight='bold',
        fontsize=10
    )
plt.xticks(rotation=15)
plt.tight_layout()
import os

plt.tight_layout()

# --------------------------------------------------
# Create results directory automatically
# --------------------------------------------------

PROJECT_ROOT = os.getcwd()

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "ql_vs_baselines.png"
)

plt.savefig(
    SAVE_PATH,
    bbox_inches="tight",
    dpi=150
)

plt.show()

print(f"✅ Comparison chart saved!")
print(f"Saved at: {SAVE_PATH}")

In [ ]:
best_baseline = max(
    v for k, v in comparison.items()
    if k != 'Q-Learning'
)
ql_revenue = comparison['Q-Learning']
improvement = (ql_revenue - best_baseline) / \
               best_baseline * 100

print("╔══════════════════════════════════════════╗")
print("║    WEEK 1 DAY 3 — Q-LEARNING SUMMARY     ║")
print("╠══════════════════════════════════════════╣")
print("║  ALGORITHM: Tabular Q-Learning           ║")
print(f"║  Episodes  : 5,000 training              ║")
print(f"║  α (lr)    : 0.10                        ║")
print(f"║  γ (disc)  : 0.99                        ║")
print(f"║  ε decay   : 1.0 → 0.01                 ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Best Baseline : ${best_baseline:<22.0f} ║")
print(f"║  Q-Learning    : ${ql_revenue:<22.0f} ║")
print(f"║  Improvement   : {improvement:+.1f}%"
      f"{'':<22} ║")
print("╠══════════════════════════════════════════╣")
if improvement > 0:
    print("║  ✅ Q-Learning beats baselines!          ║")
else:
    print("║  ⚠️  Need more training! (DQN next)      ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Q-Table Analysis 📊          ║")
print("╚══════════════════════════════════════════╝")